# GSO–Graph Cut: experiments reported in the manuscript

This notebook runs the manuscript experiments only. All saved execution outputs have been cleared. Set `DATASET_DIR`, `SECOND_DATASET_DIR`, and provide the original external `GSO` module before execution. The original optimizer implementation is not bundled in the supplied notebooks.

**Reproducibility note:** The original notebook reuses the final tuned parameters for the baseline/ablation variants, while the manuscript describes separately optimized parameters for each. This difference is not silently changed.

In [ ]:
import os

import glob

import math

import random

import json

import time

from collections import defaultdict

from time import perf_counter

import numpy as np

import cv2

import maxflow

import matplotlib.pyplot as plt

import pandas as pd

from GSO import GSO

try:
    from scipy.stats import wilcoxon
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

try:
    import nibabel as nib
    NIBABEL_AVAILABLE = True
except ImportError:
    NIBABEL_AVAILABLE = False
    print("Warning: nibabel not installed. BraTS dataset will not work.")
    print("Install with: pip install nibabel")

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

DATASET_DIR = r"/Users/app/Documents/third_paper/lgg-mri-segmentation/kaggle_3m"

SECOND_DATASET_DIR = r"/Users/app/Documents/third_paper/BraTS2020"

BRATS_MODALITY = "t2"

BRATS_OPTIMIZATION = True

BRATS_OPT_SUBSET_SIZE = 50

IMAGE_SIZE = (128, 128)

OPT_SUBSET_SIZE = 100

TRAIN_RATIO = 0.8

HARD_CONSTRAINT = 1e6

SOL_COUNT = 30

ITERATIONS_COUNT = 50

MUTATION_RATE = 0.5

EXPERIMENT_NAME = time.strftime("graphcut_gso_fullstudy_%Y-%m-%d_%H-%M-%S")

EXPERIMENT_DIR = os.path.join("experiment_logs", EXPERIMENT_NAME)

os.makedirs(EXPERIMENT_DIR, exist_ok=True)

def preprocess_brats_dataset(brats_dir, output_dir, target_size=IMAGE_SIZE, use_modality="flair"):
    if not NIBABEL_AVAILABLE:
        print("nibabel not installed. Cannot preprocess BraTS.")
        return False

    print(f"Preprocessing BraTS dataset from {brats_dir} to {output_dir}")
    os.makedirs(output_dir, exist_ok=True)

    total_slices = 0
    total_patients = 0

    subset_roots = [
        os.path.join(brats_dir, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData"),
        os.path.join(brats_dir, "BraTS2020_ValidationData", "MICCAI_BraTS2020_ValidationData"),
    ]

    for subset_root in subset_roots:
        if not os.path.exists(subset_root):
            print(f"Warning: {subset_root} not found, skipping...")
            continue

        print(f"\nScanning: {subset_root}")

        patient_dirs = [
            d for d in glob.glob(os.path.join(subset_root, "*"))
            if os.path.isdir(d)
        ]
        patient_dirs = sorted(patient_dirs)

        print(f"Found {len(patient_dirs)} patient folders")

        for patient_dir in patient_dirs:
            patient_id = os.path.basename(patient_dir)

            all_nii = glob.glob(os.path.join(patient_dir, "*.nii")) + glob.glob(os.path.join(patient_dir, "*.nii.gz"))
            all_nii_names = [os.path.basename(x).lower() for x in all_nii]

            modality_file = None
            for fpath, fname in zip(all_nii, all_nii_names):
                if f"_{use_modality}.nii" in fname or f"_{use_modality}.nii.gz" in fname:
                    modality_file = fpath
                    break

            seg_file = None
            for fpath, fname in zip(all_nii, all_nii_names):
                if "_seg.nii" in fname or "_seg.nii.gz" in fname:
                    seg_file = fpath
                    break

            if modality_file is None:
                print(f"  Warning: Missing {use_modality} file for {patient_id}")
                continue

          
            if seg_file is None:
                print(f"  Skipping {patient_id}: no segmentation mask found")
                continue

            try:
                print(f"  Processing {patient_id}...")

                modality_img = nib.load(modality_file).get_fdata()
                seg_img = nib.load(seg_file).get_fdata()

                if modality_img.shape != seg_img.shape:
                    print(f"  Warning: shape mismatch for {patient_id}, skipping")
                    continue

                out_patient_dir = os.path.join(output_dir, f"BraTS_{patient_id}")
                os.makedirs(out_patient_dir, exist_ok=True)

                num_slices = modality_img.shape[2]
                slices_saved = 0

                for slice_idx in range(num_slices):
                    modality_slice = modality_img[:, :, slice_idx]
                    seg_slice = seg_img[:, :, slice_idx]

                    if np.sum(modality_slice) == 0:
                        continue

                    if modality_slice.max() > modality_slice.min():
                        modality_slice = (
                            (modality_slice - modality_slice.min())
                            / (modality_slice.max() - modality_slice.min())
                            * 255.0
                        )
                    else:
                        modality_slice = np.zeros_like(modality_slice)

                    modality_slice = modality_slice.astype(np.uint8)

                    modality_rgb = np.stack([modality_slice] * 3, axis=-1)
                    seg_binary = (seg_slice > 0).astype(np.uint8) * 255

                    if target_size is not None:
                        modality_rgb = cv2.resize(
                            modality_rgb,
                            target_size,
                            interpolation=cv2.INTER_LINEAR
                        )
                        seg_binary = cv2.resize(
                            seg_binary,
                            target_size,
                            interpolation=cv2.INTER_NEAREST
                        )

                    slice_name = f"slice_{slice_idx:03d}.tif"
                    mask_name = f"slice_{slice_idx:03d}_mask.tif"

                    cv2.imwrite(
                        os.path.join(out_patient_dir, slice_name),
                        cv2.cvtColor(modality_rgb, cv2.COLOR_RGB2BGR)
                    )
                    cv2.imwrite(
                        os.path.join(out_patient_dir, mask_name),
                        seg_binary
                    )

                    slices_saved += 1
                    total_slices += 1

                if slices_saved > 0:
                    total_patients += 1
                    print(f"    Saved {slices_saved} slices")

            except Exception as e:
                print(f"  Error processing {patient_id}: {e}")
                continue

    print(f"\nPreprocessing complete! Total slices saved: {total_slices}")
    print(f"Patients successfully converted: {total_patients}")
    print(f"Converted files saved to: {output_dir}")

    return total_slices > 0

def preprocess_brats_enhanced(brats_dir, output_dir, target_size=IMAGE_SIZE):
    if not NIBABEL_AVAILABLE:
        print("nibabel not installed. Cannot preprocess BraTS.")
        return False

    print(f"\n{'='*60}")
    print("Enhanced BraTS Preprocessing (Multi-Modal)")
    print('='*60)
    print(f"Input: {brats_dir}")
    print(f"Output: {output_dir}")
    os.makedirs(output_dir, exist_ok=True)

    total_slices = 0
    total_patients = 0

    
    subset_root = os.path.join(brats_dir, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")
    
    if not os.path.exists(subset_root):
        print(f"Error: {subset_root} not found")
        return False

    patient_dirs = [d for d in glob.glob(os.path.join(subset_root, "*")) if os.path.isdir(d)]
    print(f"\nFound {len(patient_dirs)} patients in TrainingData")
    
    for idx, patient_dir in enumerate(patient_dirs):
        patient_id = os.path.basename(patient_dir)
        print(f"\nProcessing [{idx+1}/{len(patient_dirs)}] {patient_id}...")
        
      
        all_nii = glob.glob(os.path.join(patient_dir, "*.nii*"))
        modalities = {}
        seg_file = None
        
        for fpath in all_nii:
            fname = os.path.basename(fpath).lower()
            if "flair" in fname:
                modalities['flair'] = fpath
            elif "t2" in fname:
                modalities['t2'] = fpath
            elif "t1ce" in fname:
                modalities['t1ce'] = fpath
            elif "seg" in fname:
                seg_file = fpath

        if len(modalities) < 3:
            print(f"  Warning: Missing modalities. Found: {list(modalities.keys())}")
            continue
            
        if seg_file is None:
            print(f"  Warning: No segmentation file found")
            continue

        try:
            print(f"  Loading modalities...")
            flair = nib.load(modalities['flair']).get_fdata()
            t2 = nib.load(modalities['t2']).get_fdata()
            t1ce = nib.load(modalities['t1ce']).get_fdata()
            seg = nib.load(seg_file).get_fdata()
            
            print(f"  Volume shape: {flair.shape}")

            out_dir = os.path.join(output_dir, f"BraTS_{patient_id}")
            os.makedirs(out_dir, exist_ok=True)

            num_slices = flair.shape[2]
            slices_saved = 0

            for slice_idx in range(num_slices):
                
                flair_slice = flair[:, :, slice_idx]
                t2_slice = t2[:, :, slice_idx]
                t1ce_slice = t1ce[:, :, slice_idx]
                seg_slice = seg[:, :, slice_idx]

            
                if flair_slice.max() == 0:
                    continue

                
                def normalize(slice_data):
                    if slice_data.max() > slice_data.min():
                        return ((slice_data - slice_data.min()) / 
                               (slice_data.max() - slice_data.min()) * 255).astype(np.uint8)
                    return np.zeros_like(slice_data, dtype=np.uint8)

                flair_norm = normalize(flair_slice)
                t2_norm = normalize(t2_slice)
                t1ce_norm = normalize(t1ce_slice)
               
                rgb_img = np.stack([flair_norm, t2_norm, t1ce_norm], axis=-1)
           
                seg_binary = (seg_slice > 0).astype(np.uint8) * 255

                if target_size:
                    rgb_img = cv2.resize(rgb_img, target_size, interpolation=cv2.INTER_LINEAR)
                    seg_binary = cv2.resize(seg_binary, target_size, interpolation=cv2.INTER_NEAREST)

               
                slice_name = f"slice_{slice_idx:03d}.tif"
                mask_name = f"slice_{slice_idx:03d}_mask.tif"
                
                cv2.imwrite(os.path.join(out_dir, slice_name), 
                           cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR))
                cv2.imwrite(os.path.join(out_dir, mask_name), seg_binary)

                slices_saved += 1
                total_slices += 1

            if slices_saved > 0:
                total_patients += 1
                print(f"    Saved {slices_saved} slices")

        except Exception as e:
            print(f"  Error: {e}")
            continue

    print(f"\n{'='*60}")
    print(f"Preprocessing Complete!")
    print(f"Total patients: {total_patients}")
    print(f"Total slices saved: {total_slices}")
    print(f"Output: {output_dir}")
    print('='*60)
    
    return total_slices > 0

def get_preprocessed_brats_path(brats_dir):
    enhanced_dir = os.path.join(os.path.dirname(brats_dir), "brats_enhanced")
    
    if os.path.exists(enhanced_dir):
        pairs = collect_image_mask_pairs(enhanced_dir)
        if len(pairs) > 0:
            print(f"Found enhanced BraTS data at {enhanced_dir} with {len(pairs)} slices")
            return enhanced_dir
    
    print("Creating enhanced BraTS dataset (multi-modal)...")
    success = preprocess_brats_enhanced(brats_dir, enhanced_dir, IMAGE_SIZE)
    if success:
        return enhanced_dir
    
    print("Enhanced preprocessing failed, falling back to basic preprocessing...")
    return None

    if os.path.exists(preprocessed_dir):
        pairs = collect_image_mask_pairs(preprocessed_dir)
        if len(pairs) > 0:
            print(f"Found existing preprocessed BraTS data at {preprocessed_dir} with {len(pairs)} slices")
            return preprocessed_dir

    print("No preprocessed BraTS data found. Running preprocessing...")
    success = preprocess_brats_dataset(
        brats_dir,
        preprocessed_dir,
        target_size=IMAGE_SIZE,
        use_modality=BRATS_MODALITY
    )

    if success:
        pairs = collect_image_mask_pairs(preprocessed_dir)
        if len(pairs) > 0:
            return preprocessed_dir

    return None

def optimize_for_brats(initial_params, brats_dir, output_dir=None):
    print("\n" + "="*60)
    print("Starting BraTS-specific optimization (Starting from LGG params)")
    print("="*60)
    
    processed_brats = get_preprocessed_brats_path(brats_dir)
    if processed_brats is None:
        print("Cannot optimize: BraTS preprocessing failed")
        return initial_params
    
    all_brats_pairs = collect_image_mask_pairs(processed_brats)
    print(f"Total BraTS slices: {len(all_brats_pairs)}")
    
    brats_pos, brats_neg = split_positive_negative_pairs(all_brats_pairs)
    print(f"BraTS positive: {len(brats_pos)}, negative: {len(brats_neg)}")
    
    if len(brats_pos) == 0:
        print("No positive samples found in BraTS")
        return initial_params
    

    random.shuffle(brats_pos)
    brats_opt_pairs = brats_pos[:min(BRATS_OPT_SUBSET_SIZE, len(brats_pos))]
    brats_val_pairs = brats_pos[min(BRATS_OPT_SUBSET_SIZE, len(brats_pos)):min(BRATS_OPT_SUBSET_SIZE*2, len(brats_pos))]
    
    print(f"Using {len(brats_opt_pairs)} slices for optimization, {len(brats_val_pairs)} for validation")
    
  
    print(f"\nInitial LGG params Dice on validation set:")
    initial_val_dice = evaluate_brats_with_params(brats_val_pairs, initial_params)
    print(f"  LGG params Dice: {initial_val_dice:.4f}")
    
    lower_bound = [max(0.01, initial_params[i] * 0.5) for i in range(6)]
    upper_bound = [min(50.0, initial_params[i] * 2.0) for i in range(6)]
    
    print(f"\nNarrowed parameter bounds around LGG params:")
    param_names = ['lambda', 'gamma', 'beta', 'sigmaFG', 'sigmaBG', 'alpha']
    for i, name in enumerate(param_names):
        print(f"  {name}: [{lower_bound[i]:.4f}, {upper_bound[i]:.4f}]")
    
   
    def fitness_func_brats(solution):
        dices = []
        for image_path, mask_path in brats_opt_pairs:
            try:
                img, gt = load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE)
                gray = rgb_to_working_gray(img)
                pred = graph_cut_segment_brats(gray, solution)
                dices.append(dice_score(gt, pred))
            except Exception as e:
                continue
    
        if len(dices) == 0:
            return 1.0
    
        mean_dice = float(np.mean(dices))
        return 1.0 - mean_dice

    print("\nStarting GSO optimization for BraTS (starting from LGG params)...")
    
    result = GSO(
        fitness_func=fitness_func_brats,
        sol_count=SOL_COUNT,
        dimensions=6,
        iterations_count=ITERATIONS_COUNT,
        lower_bound=lower_bound,
        upper_bound=upper_bound,
        mutation_rate=MUTATION_RATE,
        verbose=True
    )
    
    if result.leader_fitness > 1.0 - initial_val_dice:
        print(f"\nUsing LGG params as initial best solution (Dice: {initial_val_dice:.4f})")
        result.leader_solution = initial_params
        result.leader_fitness = 1.0 - initial_val_dice
    
    brats_params = result.leader_solution
    brats_fitness = result.leader_fitness
    brats_dice = 1.0 - brats_fitness
    
    print("\n" + "="*60)
    print("BraTS Optimization Complete")
    print("="*60)
    print(f"Best Dice on BraTS subset: {brats_dice:.4f}")
    print("Best params for BraTS:")
    print(f"  lambda: {brats_params[0]:.4f}")
    print(f"  gamma: {brats_params[1]:.4f}")
    print(f"  beta: {brats_params[2]:.6f}")
    print(f"  sigmaFG: {brats_params[3]:.4f}")
    print(f"  sigmaBG: {brats_params[4]:.4f}")
    print(f"  alpha: {brats_params[5]:.4f}")
    
   
    print("\nValidation on held-out set:")
    val_dices = []
    for img_path, mask_path in brats_val_pairs:
        try:
            img, gt = load_image_mask(img_path, mask_path)
            gray = rgb_to_working_gray(img)
            pred = graph_cut_segment_brats(gray, brats_params)
            val_dices.append(dice_score(gt, pred))
        except:
            continue
    
    val_dice = np.mean(val_dices) if val_dices else 0
    print(f"  Validation Dice: {val_dice:.4f}")
    

    print("\nComparison with LGG-optimized params:")
    print(f"  LGG params Dice: {initial_val_dice:.4f}")
    print(f"  BraTS params Dice: {brats_dice:.4f}")
    print(f"  Validation Dice: {val_dice:.4f}")
    print(f"  Improvement: {(val_dice - initial_val_dice)*100:.2f}%")
    
 
    if output_dir:
        np.save(os.path.join(output_dir, "brats_optimized_params.npy"), brats_params)
        with open(os.path.join(output_dir, "brats_optimized_params.txt"), "w") as f:
            f.write(f"BraTS Optimized Parameters\n")
            f.write(f"==========================\n")
            f.write(f"lambda: {brats_params[0]}\n")
            f.write(f"gamma: {brats_params[1]}\n")
            f.write(f"beta: {brats_params[2]}\n")
            f.write(f"sigmaFG: {brats_params[3]}\n")
            f.write(f"sigmaBG: {brats_params[4]}\n")
            f.write(f"alpha: {brats_params[5]}\n")
            f.write(f"\nValidation Dice: {val_dice:.4f}\n")
            f.write(f"LGG params Dice: {initial_val_dice:.4f}\n")
    
    return brats_params

def evaluate_brats_with_params(pairs_list, params):
    dices = []
    for image_path, mask_path in pairs_list[:min(50, len(pairs_list))]:
        try:
            img, gt = load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE)
            gray = rgb_to_working_gray(img)
            
            pred = graph_cut_segment_brats(gray, params) 
            dices.append(dice_score(gt, pred))
        except:
            continue
    return float(np.mean(dices)) if dices else 0.0

def evaluate_brats_full(brats_dir, params, output_dir=None):
    print("\n" + "="*60)
    print("Full BraTS Dataset Evaluation")
    print("="*60)
    
    processed_brats = get_preprocessed_brats_path(brats_dir)
    if processed_brats is None:
        return None, None, None
    
    all_pairs = collect_image_mask_pairs(processed_brats)
    pos_pairs, neg_pairs = split_positive_negative_pairs(all_pairs)
    
    print(f"Total BraTS slices: {len(all_pairs)}")
    print(f"  Positive: {len(pos_pairs)}")
    print(f"  Negative: {len(neg_pairs)}")
    

    pos_metrics = evaluate_pairs(pos_pairs, params, segment_fn=graph_cut_segment_brats) 
    neg_metrics = evaluate_negative_pairs(neg_pairs, params, segment_fn=graph_cut_segment_brats) 
    _, pos_df = evaluate_pairs_detailed(pos_pairs, params, segment_fn=graph_cut_segment_brats)
    
    print("\nBraTS Positive Metrics:")
    for k, v in pos_metrics.items():
        print(f"  {k}: {v}")
    
    print("\nBraTS Negative Metrics:")
    for k, v in neg_metrics.items():
        print(f"  {k}: {v}")
    
    # Save results
    if output_dir:
        pos_df.to_csv(os.path.join(output_dir, "brats_full_evaluation.csv"), index=False)
        
        # Save summary
        summary = {
            "positive_metrics": pos_metrics,
            "negative_metrics": neg_metrics,
            "params_used": params.tolist()
        }
        with open(os.path.join(output_dir, "brats_evaluation_summary.json"), "w") as f:
            json.dump(summary, f, indent=4)
    
    return pos_metrics, neg_metrics, pos_df

def collect_image_mask_pairs(dataset_dir):
    image_paths = []
    mask_paths = []

    all_tif = glob.glob(os.path.join(dataset_dir, "**", "*.tif"), recursive=True)

    for p in all_tif:
        fname = os.path.basename(p).lower()
        if fname.endswith("_mask.tif"):
            continue

        mask_p = p.replace(".tif", "_mask.tif")
        if os.path.exists(mask_p):
            image_paths.append(p)
            mask_paths.append(mask_p)

    pairs = list(zip(image_paths, mask_paths))
    pairs.sort()
    return pairs

def get_patient_id_from_path(image_path):
    return os.path.basename(os.path.dirname(image_path))

def split_by_patient(pairs, train_ratio=0.8, seed=42):
    patient_to_pairs = defaultdict(list)

    for image_path, mask_path in pairs:
        patient_id = get_patient_id_from_path(image_path)
        patient_to_pairs[patient_id].append((image_path, mask_path))

    patient_ids = sorted(patient_to_pairs.keys())

    rng = random.Random(seed)
    rng.shuffle(patient_ids)

    split_idx = int(len(patient_ids) * train_ratio)

    train_patient_ids = patient_ids[:split_idx]
    val_patient_ids = patient_ids[split_idx:]

    train_pairs = []
    val_pairs = []

    for pid in train_patient_ids:
        train_pairs.extend(patient_to_pairs[pid])

    for pid in val_patient_ids:
        val_pairs.extend(patient_to_pairs[pid])

    return train_pairs, val_pairs, train_patient_ids, val_patient_ids

def load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE):
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError(f"Could not read image: {image_path}")
    if mask is None:
        raise ValueError(f"Could not read mask: {mask_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    if target_size is not None:
        img = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)

    mask = (mask > 0).astype(np.uint8)
    return img, mask

def split_positive_negative_pairs(pairs_list):
    positive_pairs = []
    negative_pairs = []

    for image_path, mask_path in pairs_list:
        _, gt = load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE)
        if gt.sum() > 0:
            positive_pairs.append((image_path, mask_path))
        else:
            negative_pairs.append((image_path, mask_path))

    return positive_pairs, negative_pairs

def rgb_to_working_gray(img_rgb):
    channel = img_rgb[:, :, 1].astype(np.float32)

    cmin, cmax = channel.min(), channel.max()
    if cmax > cmin:
        channel = 255.0 * (channel - cmin) / (cmax - cmin)
    else:
        channel = np.zeros_like(channel)

    return channel.astype(np.float32)

def largest_connected_component(mask_u8):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)

    if num_labels <= 1:
        return mask_u8

    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    out = np.zeros_like(mask_u8)
    out[labels == largest_label] = 1
    return out.astype(np.uint8)

def remove_small_components(mask_u8, min_area=20):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    out = np.zeros_like(mask_u8)

    for lbl in range(1, num_labels):
        area = stats[lbl, cv2.CC_STAT_AREA]
        if area >= min_area:
            out[labels == lbl] = 1

    return out.astype(np.uint8)

def fill_holes(mask_u8):
    mask255 = (mask_u8 * 255).astype(np.uint8)
    h, w = mask255.shape

    flood = mask255.copy()
    flood_mask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, flood_mask, (0, 0), 255)

    flood_inv = cv2.bitwise_not(flood)
    filled = cv2.bitwise_or(mask255, flood_inv)

    return (filled > 0).astype(np.uint8)

def remove_border_components(mask_u8):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    h, w = mask_u8.shape
    out = np.zeros_like(mask_u8)

    for lbl in range(1, num_labels):
        ys, xs = np.where(labels == lbl)
        if ys.size == 0:
            continue

        touches_border = (
            np.any(ys == 0) or np.any(ys == h - 1) or
            np.any(xs == 0) or np.any(xs == w - 1)
        )

        if not touches_border:
            out[labels == lbl] = 1

    return out.astype(np.uint8)

def component_compactness(component_mask):
    component_mask = (component_mask > 0).astype(np.uint8)
    area = float(component_mask.sum())
    if area == 0:
        return 0.0

    contours, _ = cv2.findContours(component_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0.0

    perim = float(sum(cv2.arcLength(cnt, True) for cnt in contours))
    if perim <= 1e-6:
        return 0.0

    return float((4.0 * np.pi * area) / (perim * perim))

def keep_best_tumor_component(mask_u8, min_area=12):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)

    if num_labels <= 1:
        return np.zeros_like(mask_u8, dtype=np.uint8)

    best_score = -1.0
    best_mask = np.zeros_like(mask_u8, dtype=np.uint8)

    for lbl in range(1, num_labels):
        area = int(stats[lbl, cv2.CC_STAT_AREA])
        if area < min_area:
            continue

        comp = (labels == lbl).astype(np.uint8)

        ys, xs = np.where(comp > 0)
        if ys.size == 0:
            continue

        h, w = mask_u8.shape
        touches_border = (
            np.any(ys == 0) or np.any(ys == h - 1) or
            np.any(xs == 0) or np.any(xs == w - 1)
        )
        if touches_border:
            continue

        compact = component_compactness(comp)
        score = area * compact

        if score > best_score:
            best_score = score
            best_mask = comp.copy()

    return best_mask.astype(np.uint8)

def get_brain_mask(gray):
    gray_u8 = np.clip(gray, 0, 255).astype(np.uint8)

    _, mask = cv2.threshold(gray_u8, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    mask = largest_connected_component(mask)

    kernel5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel5, iterations=2)
    mask = fill_holes(mask)
    mask = largest_connected_component(mask)

    erode_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.erode(mask, erode_k, iterations=1)

    return mask.astype(np.uint8)

def dice_score(y_true, y_pred):
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)

    inter = np.logical_and(y_true, y_pred).sum()
    denom = y_true.sum() + y_pred.sum()

    if denom == 0:
        return 1.0
    return (2.0 * inter) / denom

def iou_score(y_true, y_pred):
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)

    inter = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()

    if union == 0:
        return 1.0
    return inter / union

def confusion_counts(y_true, y_pred):
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)

    tp = np.logical_and(y_true, y_pred).sum()
    tn = np.logical_and(~y_true, ~y_pred).sum()
    fp = np.logical_and(~y_true, y_pred).sum()
    fn = np.logical_and(y_true, ~y_pred).sum()

    return tp, tn, fp, fn

def precision_score_bin(y_true, y_pred):
    tp, _, fp, _ = confusion_counts(y_true, y_pred)
    denom = tp + fp
    if denom == 0:
        return 1.0
    return tp / denom

def recall_score_bin(y_true, y_pred):
    tp, _, _, fn = confusion_counts(y_true, y_pred)
    denom = tp + fn
    if denom == 0:
        return 1.0
    return tp / denom

def f1_score_bin(y_true, y_pred):
    p = precision_score_bin(y_true, y_pred)
    r = recall_score_bin(y_true, y_pred)
    denom = p + r
    if denom == 0:
        return 0.0
    return 2.0 * p * r / denom

def specificity_score_bin(y_true, y_pred):
    _, tn, fp, _ = confusion_counts(y_true, y_pred)
    denom = tn + fp
    if denom == 0:
        return 1.0
    return tn / denom

def generate_auto_seeds(gray, alpha):
    gray_u8 = np.clip(gray, 0, 255).astype(np.uint8)

    brain_mask = get_brain_mask(gray_u8)

    erode_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    brain_inner = cv2.erode(brain_mask, erode_k, iterations=1)

    brain_vals = gray_u8[brain_inner > 0]
    if brain_vals.size == 0:
        brain_vals = gray_u8[brain_mask > 0]

    if brain_vals.size == 0:
        fg_seed = np.zeros_like(gray_u8, dtype=np.uint8)
        bg_seed = np.zeros_like(gray_u8, dtype=np.uint8)
        return fg_seed, bg_seed, brain_mask

    mean_val = float(np.mean(brain_vals))
    std_val = float(np.std(brain_vals))

    fg_thr_a = mean_val + alpha * std_val
    fg_thr_b = np.percentile(brain_vals, 97)
    fg_thr = max(fg_thr_a, fg_thr_b)

    fg_seed = ((gray_u8 >= fg_thr) & (brain_inner > 0)).astype(np.uint8)
    fg_seed = remove_small_components(fg_seed, min_area=12)

    if fg_seed.sum() < 8:
        fg_thr = np.percentile(brain_vals, 99)
        fg_seed = ((gray_u8 >= fg_thr) & (brain_inner > 0)).astype(np.uint8)
        fg_seed = remove_small_components(fg_seed, min_area=8)

    bg_core = cv2.erode(brain_mask, erode_k, iterations=2)
    fg_dil = cv2.dilate(fg_seed, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)), iterations=1)

    bg_seed = ((bg_core > 0) & (fg_dil == 0)).astype(np.uint8)

    bg_thr = np.percentile(brain_vals, 70)
    bg_seed = ((bg_seed > 0) & (gray_u8 <= bg_thr)).astype(np.uint8)

    bg_seed = np.maximum(bg_seed, (brain_mask == 0).astype(np.uint8)).astype(np.uint8)

    overlap = (fg_seed == 1) & (bg_seed == 1)
    fg_seed[overlap] = 0
    bg_seed[overlap] = 0

    return fg_seed, bg_seed, brain_mask

def generate_auto_seeds_baseline(gray, alpha):
    gray_u8 = np.clip(gray, 0, 255).astype(np.uint8)

    otsu_val, _ = cv2.threshold(gray_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    std_val = float(np.std(gray_u8))
    margin = max(1.0, alpha * std_val)

    fg_thr = min(255.0, otsu_val + margin)
    bg_thr = max(0.0, otsu_val - margin)

    fg_seed = (gray_u8 >= fg_thr).astype(np.uint8)
    bg_seed = (gray_u8 <= bg_thr).astype(np.uint8)

    kernel = np.ones((3, 3), np.uint8)
    fg_seed = cv2.morphologyEx(fg_seed, cv2.MORPH_OPEN, kernel, iterations=1)
    bg_seed = cv2.morphologyEx(bg_seed, cv2.MORPH_OPEN, kernel, iterations=1)

    overlap = (fg_seed == 1) & (bg_seed == 1)
    fg_seed[overlap] = 0
    bg_seed[overlap] = 0

    if fg_seed.sum() < 10:
        fg_seed = (gray_u8 >= np.percentile(gray_u8, 95)).astype(np.uint8)
    if bg_seed.sum() < 10:
        bg_seed = (gray_u8 <= np.percentile(gray_u8, 5)).astype(np.uint8)

    overlap = (fg_seed == 1) & (bg_seed == 1)
    fg_seed[overlap] = 0
    bg_seed[overlap] = 0

    brain_mask = np.ones_like(gray_u8, dtype=np.uint8)
    return fg_seed, bg_seed, brain_mask

def generate_auto_seeds_brats(gray, alpha):
    gray_u8 = np.clip(gray, 0, 255).astype(np.uint8)
    
    _, brain_mask1 = cv2.threshold(gray_u8, 1, 1, cv2.THRESH_BINARY)
    _, brain_mask2 = cv2.threshold(gray_u8, 5, 1, cv2.THRESH_BINARY)
    brain_mask = cv2.bitwise_or(brain_mask1, brain_mask2)
    brain_mask = largest_connected_component(brain_mask)
    
    if brain_mask.sum() == 0:
        brain_mask = np.ones_like(gray_u8)
    
    brain_vals = gray_u8[brain_mask > 0]
    if brain_vals.size == 0:
        brain_vals = gray_u8.flatten()
    

    fg_thr1 = np.percentile(brain_vals, 85) if brain_vals.size > 0 else 200
    
    if brain_vals.size > 0:
        fg_thr2 = np.mean(brain_vals) + alpha * np.std(brain_vals)
    else:
        fg_thr2 = 200
    
    fg_thr = min(fg_thr1, fg_thr2)
    
    fg_seed = ((gray_u8 >= fg_thr) & (brain_mask > 0)).astype(np.uint8)
    
    if fg_seed.sum() < 5 and brain_vals.size > 0:
        fg_thr = np.percentile(brain_vals, 95)
        fg_seed = ((gray_u8 >= fg_thr) & (brain_mask > 0)).astype(np.uint8)
    
    fg_seed = remove_small_components(fg_seed, min_area=5)
    
    bg_seed = ((brain_mask > 0) & (fg_seed == 0)).astype(np.uint8)
    bg_seed = np.maximum(bg_seed, (brain_mask == 0).astype(np.uint8))
    
    return fg_seed, bg_seed, brain_mask

def gaussian_nll(x, mu, sigma):
    sigma = max(float(sigma), 1e-6)
    return 0.5 * np.log(2.0 * np.pi * sigma * sigma) + ((x - mu) ** 2) / (2.0 * sigma * sigma)

def _graph_cut_segment_core(gray, params, mode="final"):
    lam = float(params[0])
    gamma = float(params[1])
    beta = float(params[2])
    sigma_fg = float(params[3])
    sigma_bg = float(params[4])
    alpha = float(params[5])

    h, w = gray.shape

    if mode == "baseline":
        fg_seed, bg_seed, brain_mask = generate_auto_seeds_baseline(gray, alpha)
        use_brain_constraint = False
        use_fill = False
        use_border_remove = False
        use_best_component = False
    elif mode == "brain_mask":
        fg_seed, bg_seed, brain_mask = generate_auto_seeds(gray, alpha)
        use_brain_constraint = True
        use_fill = False
        use_border_remove = False
        use_best_component = False
    elif mode == "postprocess":
        fg_seed, bg_seed, brain_mask = generate_auto_seeds(gray, alpha)
        use_brain_constraint = True
        use_fill = True
        use_border_remove = True
        use_best_component = False
    else:
        fg_seed, bg_seed, brain_mask = generate_auto_seeds(gray, alpha)
        use_brain_constraint = True
        use_fill = True
        use_border_remove = True
        use_best_component = True

    fg_vals = gray[fg_seed > 0]
    bg_vals = gray[bg_seed > 0]

    if fg_vals.size == 0:
        fg_mean = float(np.percentile(gray[brain_mask > 0], 95)) if np.any(brain_mask > 0) else float(np.percentile(gray, 95))
    else:
        fg_mean = float(np.mean(fg_vals))

    if bg_vals.size == 0:
        bg_mean = float(np.percentile(gray[brain_mask > 0], 30)) if np.any(brain_mask > 0) else float(np.percentile(gray, 30))
    else:
        bg_mean = float(np.mean(bg_vals))

    d_fg = lam * gaussian_nll(gray, fg_mean, sigma_fg)
    d_bg = lam * gaussian_nll(gray, bg_mean, sigma_bg)

    d_fg = d_fg.copy()
    d_bg = d_bg.copy()

    d_fg[fg_seed == 1] = 0.0
    d_bg[fg_seed == 1] = HARD_CONSTRAINT

    d_bg[bg_seed == 1] = 0.0
    d_fg[bg_seed == 1] = HARD_CONSTRAINT

    if use_brain_constraint:
        d_fg[brain_mask == 0] = HARD_CONSTRAINT
        d_bg[brain_mask == 0] = 0.0

    g = maxflow.Graph[float]()
    nodes = g.add_grid_nodes((h, w))

    for y in range(h):
        for x in range(w):
            n = nodes[y, x]
            g.add_tedge(n, float(d_bg[y, x]), float(d_fg[y, x]))

    for y in range(h):
        for x in range(w):
            p = nodes[y, x]
            I_p = gray[y, x]

            if x + 1 < w:
                q = nodes[y, x + 1]
                I_q = gray[y, x + 1]
                diff = float(I_p - I_q)
                w_pq = gamma * math.exp(-beta * (diff * diff))
                g.add_edge(p, q, w_pq, w_pq)

            if y + 1 < h:
                q = nodes[y + 1, x]
                I_q = gray[y + 1, x]
                diff = float(I_p - I_q)
                w_pq = gamma * math.exp(-beta * (diff * diff))
                g.add_edge(p, q, w_pq, w_pq)

    g.maxflow()

    seg = np.zeros((h, w), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            seg[y, x] = 1 if g.get_segment(nodes[y, x]) == 0 else 0

    if use_brain_constraint:
        seg = (seg * brain_mask).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    seg = cv2.morphologyEx(seg, cv2.MORPH_OPEN, kernel, iterations=1)
    seg = cv2.morphologyEx(seg, cv2.MORPH_CLOSE, kernel, iterations=1)

    if use_fill:
        seg = fill_holes(seg)

    seg = remove_small_components(seg, min_area=12)

    if use_border_remove:
        seg = remove_border_components(seg)

    if use_best_component:
        seg_best = keep_best_tumor_component(seg, min_area=12)
        if seg_best.sum() > 0:
            seg = seg_best
        else:
            seg = np.zeros_like(seg, dtype=np.uint8)

    return seg

def graph_cut_segment(gray, params):
    return _graph_cut_segment_core(gray, params, mode="final")

def graph_cut_segment_brats(gray, params):
    lam, gamma, beta, sigma_fg, sigma_bg, alpha = [float(p) for p in params]
    h, w = gray.shape
    
    fg_seed, bg_seed, brain_mask = generate_auto_seeds_brats(gray, alpha)
    
    fg_vals = gray[fg_seed > 0]
    bg_vals = gray[bg_seed > 0]
    
    brain_pixels = gray[brain_mask > 0]
    if brain_pixels.size == 0:
        return np.zeros((h, w), dtype=np.uint8)
    
    fg_mean = np.mean(fg_vals) if fg_vals.size > 0 else np.percentile(brain_pixels, 90)
    bg_mean = np.mean(bg_vals) if bg_vals.size > 0 else np.percentile(brain_pixels, 30)
    
    d_fg = lam * gaussian_nll(gray, fg_mean, sigma_fg)
    d_bg = lam * gaussian_nll(gray, bg_mean, sigma_bg)

    d_fg[fg_seed == 1] = 0
    d_bg[fg_seed == 1] = HARD_CONSTRAINT
    d_bg[bg_seed == 1] = 0
    d_fg[bg_seed == 1] = HARD_CONSTRAINT
    d_fg[brain_mask == 0] = HARD_CONSTRAINT
    d_bg[brain_mask == 0] = 0
    
    g = maxflow.Graph[float]()
    nodes = g.add_grid_nodes((h, w))
    
    for y in range(h):
        for x in range(w):
            g.add_tedge(nodes[y, x], float(d_bg[y, x]), float(d_fg[y, x]))
    
    for y in range(h):
        for x in range(w):
            if x + 1 < w:
                diff = abs(float(gray[y, x] - gray[y, x + 1]))
                weight = gamma * math.exp(-beta * diff * diff)
                g.add_edge(nodes[y, x], nodes[y, x + 1], weight, weight)
            if y + 1 < h:
                diff = abs(float(gray[y, x] - gray[y + 1, x]))
                weight = gamma * math.exp(-beta * diff * diff)
                g.add_edge(nodes[y, x], nodes[y + 1, x], weight, weight)
    
    g.maxflow()
    
    seg = np.zeros((h, w), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            seg[y, x] = 1 if g.get_segment(nodes[y, x]) == 0 else 0
    
    seg = seg * brain_mask
    
    # Keep ALL components, even very small ones
    num_labels, labels = cv2.connectedComponents(seg.astype(np.uint8))
    final_seg = np.zeros_like(seg)
    for i in range(1, num_labels):
        component = (labels == i).astype(np.uint8)
        if component.sum() >= 3:  # Keep components as small as 3 pixels
            final_seg = cv2.bitwise_or(final_seg, component)
    
    return final_seg

def graph_cut_segment_baseline(gray, params):
    return _graph_cut_segment_core(gray, params, mode="baseline")

def graph_cut_segment_brainmask(gray, params):
    return _graph_cut_segment_core(gray, params, mode="brain_mask")

def graph_cut_segment_postprocess(gray, params):
    return _graph_cut_segment_core(gray, params, mode="postprocess")

def evaluate_pairs(pairs_list, params, segment_fn=graph_cut_segment):
    dices = []

    for image_path, mask_path in pairs_list:
        img, gt = load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE)
        gray = rgb_to_working_gray(img)
        pred = segment_fn(gray, params)
        dices.append(dice_score(gt, pred))

    dices = np.array(dices, dtype=np.float32)
    return {
        "mean_dice": float(np.mean(dices)) if len(dices) else None,
        "median_dice": float(np.median(dices)) if len(dices) else None,
        "min_dice": float(np.min(dices)) if len(dices) else None,
        "max_dice": float(np.max(dices)) if len(dices) else None,
        "num_samples": int(len(dices)),
    }

def evaluate_negative_pairs(pairs_list, params, segment_fn=graph_cut_segment):
    total = len(pairs_list)
    true_negative = 0
    false_positive = 0
    fp_pixels = []

    for image_path, mask_path in pairs_list:
        img, gt = load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE)
        gray = rgb_to_working_gray(img)
        pred = segment_fn(gray, params)

        pred_pixels = int(pred.sum())
        fp_pixels.append(pred_pixels)

        if pred_pixels == 0:
            true_negative += 1
        else:
            false_positive += 1

    if total == 0:
        return {
            "num_samples": 0,
            "true_negative": 0,
            "false_positive": 0,
            "true_negative_rate": None,
            "false_positive_rate": None,
            "mean_false_positive_pixels": None,
            "median_false_positive_pixels": None,
            "max_false_positive_pixels": None,
        }

    fp_pixels = np.array(fp_pixels, dtype=np.float32)

    return {
        "num_samples": total,
        "true_negative": int(true_negative),
        "false_positive": int(false_positive),
        "true_negative_rate": float(true_negative / total),
        "false_positive_rate": float(false_positive / total),
        "mean_false_positive_pixels": float(np.mean(fp_pixels)),
        "median_false_positive_pixels": float(np.median(fp_pixels)),
        "max_false_positive_pixels": float(np.max(fp_pixels)),
    }

def evaluate_pairs_detailed(pairs_list, params, segment_fn=graph_cut_segment):
    rows = []
    skipped_small = 0
    skipped_error = 0
    
    for image_path, mask_path in pairs_list:
        try:
            patient_id = get_patient_id_from_path(image_path)
            img, gt = load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE)
            
            if gt.sum() < 10:
                skipped_small += 1
                continue
                
            gray = rgb_to_working_gray(img)
            pred = segment_fn(gray, params)

            rows.append({
                "patient_id": patient_id,
                "image": image_path,
                "mask": mask_path,
                "dice": float(dice_score(gt, pred)),
                "iou": float(iou_score(gt, pred)),
                "precision": float(precision_score_bin(gt, pred)),
                "recall": float(recall_score_bin(gt, pred)),
                "f1": float(f1_score_bin(gt, pred)),
                "specificity": float(specificity_score_bin(gt, pred)),
                "gt_pixels": int(gt.sum()),
                "pred_pixels": int(pred.sum()),
            })
        except Exception as e:
            skipped_error += 1
            continue

    df = pd.DataFrame(rows)
    
    print(f"  Evaluation: {len(rows)} slices, skipped {skipped_small} tiny tumors, {skipped_error} errors")

    summary = {
        "mean_dice": float(df["dice"].mean()) if len(df) else 0,
        "median_dice": float(df["dice"].median()) if len(df) else 0,
        "min_dice": float(df["dice"].min()) if len(df) else 0,
        "max_dice": float(df["dice"].max()) if len(df) else 0,
        "mean_iou": float(df["iou"].mean()) if len(df) else 0,
        "mean_precision": float(df["precision"].mean()) if len(df) else 0,
        "mean_recall": float(df["recall"].mean()) if len(df) else 0,
        "mean_f1": float(df["f1"].mean()) if len(df) else 0,
        "mean_specificity": float(df["specificity"].mean()) if len(df) else 0,
        "num_samples": int(len(df)),
        "skipped_tiny_tumors": skipped_small
    }

    return summary, df

def wilcoxon_test_from_dfs(df_a, df_b, metric="dice"):
    if not SCIPY_AVAILABLE:
        return {
            "available": False,
            "p_value": None,
            "statistic": None,
            "note": "scipy not installed"
        }

    common = pd.merge(
        df_a[["image", metric]],
        df_b[["image", metric]],
        on="image",
        suffixes=("_a", "_b")
    )

    if len(common) == 0:
        return {
            "available": True,
            "p_value": None,
            "statistic": None,
            "note": "no common samples"
        }

    stat, p = wilcoxon(common[f"{metric}_a"], common[f"{metric}_b"])
    return {
        "available": True,
        "statistic": float(stat),
        "p_value": float(p),
        "num_samples": int(len(common))
    }

def benchmark_runtime(pairs_list, params, segment_fn, num_cases=20):
    chosen = pairs_list[:min(num_cases, len(pairs_list))]
    times = []

    for image_path, mask_path in chosen:
        img, _ = load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE)
        gray = rgb_to_working_gray(img)

        t0 = perf_counter()
        _ = segment_fn(gray, params)
        t1 = perf_counter()

        times.append(t1 - t0)

    times = np.array(times, dtype=np.float64)
    return {
        "num_cases": int(len(times)),
        "mean_seconds": float(np.mean(times)) if len(times) else None,
        "median_seconds": float(np.median(times)) if len(times) else None,
        "min_seconds": float(np.min(times)) if len(times) else None,
        "max_seconds": float(np.max(times)) if len(times) else None,
    }

def run_ablation_study(pairs_list, params):
    variants = {
        "baseline": graph_cut_segment_baseline,
        "brain_mask": graph_cut_segment_brainmask,
        "postprocess": graph_cut_segment_postprocess,
        "final": graph_cut_segment,
    }

    rows = []
    dfs = {}

    for name, fn in variants.items():
        summary, df = evaluate_pairs_detailed(pairs_list, params, segment_fn=fn)
        dfs[name] = df
        row = {"variant": name}
        row.update(summary)
        rows.append(row)

    ablation_df = pd.DataFrame(rows)
    return ablation_df, dfs

def save_ranked_examples(df_metrics, params, save_dir, pairs_lookup, n_each=3, segment_fn=graph_cut_segment):
    os.makedirs(save_dir, exist_ok=True)

    df_sorted = df_metrics.sort_values("dice", ascending=False).reset_index(drop=True)

    groups = {
        "best": df_sorted.head(n_each),
        "worst": df_sorted.tail(n_each),
        "median": df_sorted.iloc[max(0, len(df_sorted)//2 - n_each//2): max(0, len(df_sorted)//2 - n_each//2) + n_each]
    }

    for group_name, df_group in groups.items():
        for idx, row in enumerate(df_group.itertuples(index=False), start=1):
            image_path = row.image
            mask_path = pairs_lookup[image_path]

            img, gt = load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE)
            gray = rgb_to_working_gray(img)
            pred = segment_fn(gray, params)
            d = dice_score(gt, pred)

            plt.figure(figsize=(9, 3))

            plt.subplot(1, 3, 1)
            plt.imshow(img)
            plt.title("MRI")
            plt.axis("off")

            plt.subplot(1, 3, 2)
            plt.imshow(gt, cmap="gray")
            plt.title("GT")
            plt.axis("off")

            plt.subplot(1, 3, 3)
            plt.imshow(pred, cmap="gray")
            plt.title(f"Pred | Dice={d:.4f}")
            plt.axis("off")

            plt.tight_layout()
            out_path = os.path.join(save_dir, f"{group_name}_{idx}.png")
            plt.savefig(out_path, dpi=300, bbox_inches="tight")
            plt.close()

def save_false_positive_examples(negative_pairs, params, save_dir, n_examples=3, segment_fn=graph_cut_segment):
    os.makedirs(save_dir, exist_ok=True)

    rows = []
    for image_path, mask_path in negative_pairs:
        img, gt = load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE)
        gray = rgb_to_working_gray(img)
        pred = segment_fn(gray, params)
        rows.append((image_path, mask_path, int(pred.sum())))

    rows.sort(key=lambda x: x[2], reverse=True)
    chosen = rows[:min(n_examples, len(rows))]

    for idx, (image_path, mask_path, pred_pixels) in enumerate(chosen, start=1):
        img, gt = load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE)
        gray = rgb_to_working_gray(img)
        pred = segment_fn(gray, params)

        plt.figure(figsize=(9, 3))

        plt.subplot(1, 3, 1)
        plt.imshow(img)
        plt.title("MRI")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(gt, cmap="gray")
        plt.title("GT (negative)")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(pred, cmap="gray")
        plt.title(f"FP pixels={pred_pixels}")
        plt.axis("off")

        plt.tight_layout()
        out_path = os.path.join(save_dir, f"false_positive_{idx}.png")
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()

def evaluate_second_dataset(second_dataset_dir, best_params):
    if second_dataset_dir is None:
        return None, None, None, None

    if not os.path.exists(second_dataset_dir):
        print(f"Second dataset directory not found: {second_dataset_dir}")
        return None, None, None, None

    is_brats = any(x in os.path.basename(second_dataset_dir).lower() for x in ['brats', 'brats2020'])
    
    if is_brats:
        print("BraTS dataset detected. Checking for preprocessed data...")
        processed_dir = get_preprocessed_brats_path(second_dataset_dir)
        if processed_dir is None:
            print("Failed to preprocess BraTS dataset")
            return None, None, None, None
        pairs = collect_image_mask_pairs(processed_dir)
    else:
        pairs = collect_image_mask_pairs(second_dataset_dir)
    
    if len(pairs) == 0:
        print(f"No valid image/mask pairs found in {second_dataset_dir}")
        return None, None, None, None

    print(f"Found {len(pairs)} pairs in second dataset")
    
    pos_pairs, neg_pairs = split_positive_negative_pairs(pairs)
    print(f"  Positive: {len(pos_pairs)}, Negative: {len(neg_pairs)}")

    pos_metrics = evaluate_pairs(pos_pairs, best_params, segment_fn=graph_cut_segment)
    neg_metrics = evaluate_negative_pairs(neg_pairs, best_params, segment_fn=graph_cut_segment)
    all_metrics = evaluate_pairs(pairs, best_params, segment_fn=graph_cut_segment)

    _, pos_df = evaluate_pairs_detailed(pos_pairs, best_params, segment_fn=graph_cut_segment)
    return pos_metrics, neg_metrics, all_metrics, pos_df

def show_predictions(pairs_list, params, n=5, save_path=None, segment_fn=graph_cut_segment):
    n = min(n, len(pairs_list))
    chosen = random.sample(pairs_list, n)

    plt.figure(figsize=(12, 4 * n))

    idx = 1
    for image_path, mask_path in chosen:
        img, gt = load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE)
        gray = rgb_to_working_gray(img)
        pred = segment_fn(gray, params)
        d = dice_score(gt, pred)
        patient_id = get_patient_id_from_path(image_path)

        plt.subplot(n, 3, idx)
        plt.imshow(img)
        plt.title(f"MRI\n{patient_id}")
        plt.axis("off")
        idx += 1

        plt.subplot(n, 3, idx)
        plt.imshow(gt, cmap="gray")
        plt.title("GT")
        plt.axis("off")
        idx += 1

        plt.subplot(n, 3, idx)
        plt.imshow(pred, cmap="gray")
        plt.title(f"Pred | Dice={d:.4f}")
        plt.axis("off")
        idx += 1

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close()

def create_brats_sample_predictions(brats_pairs, params, n_samples=2, save_path=None):
    if brats_pairs is None or len(brats_pairs) == 0:
        print("Cannot create BraTS samples: No valid pairs.")
        return
    
    dices = []
    for img_p, mask_p in brats_pairs[:50]: 
        try:
            img, gt = load_image_mask(img_p, mask_p, target_size=IMAGE_SIZE)
            gray = rgb_to_working_gray(img)
            pred = graph_cut_segment_brats(gray, params)
            dices.append((img_p, mask_p, dice_score(gt, pred)))
        except:
            continue
    
    if not dices:
        print("Could not compute dice for any BraTS slices.")
        return
        
    dices.sort(key=lambda x: x[2])
    samples = [dices[0], dices[-1]] if len(dices) > 1 else [dices[0], dices[0]]
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    plt.suptitle("BraTS Sample Predictions", fontsize=16)
    
    for idx, (img_p, mask_p, dice_val) in enumerate(samples):
        img, gt = load_image_mask(img_p, mask_p, target_size=IMAGE_SIZE)
        gray = rgb_to_working_gray(img)
        pred = graph_cut_segment_brats(gray, params)
        patient_id = get_patient_id_from_path(img_p)
        
        imgs = [img, gt, pred]
        titles = [f"MRI: {patient_id}", f"GT (Dice={dice_val:.3f})", "Prediction"]
        
        for j, (im, title) in enumerate(zip(imgs, titles)):
            axes[idx, j].imshow(im, cmap='gray' if j>0 else None)
            axes[idx, j].set_title(title)
            axes[idx, j].axis('off')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"BraTS sample predictions saved to {save_path}")
    plt.show()
    plt.close()

def run_parameter_sensitivity_analysis(pairs_list, base_params, param_names, param_indices,
                                       variation_scales=[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0],
                                       segment_fn=graph_cut_segment):
    sensitivity_results = {}
    
    for param_idx, param_name in zip(param_indices, param_names):
        print(f"  Analyzing sensitivity for {param_name}...")
        variations = []
        mean_dices = []
        
        for scale in variation_scales:
            varied_params = base_params.copy()
            varied_params[param_idx] = base_params[param_idx] * scale
            
            metrics = evaluate_pairs(pairs_list, varied_params, segment_fn=segment_fn)
            mean_dices.append(metrics['mean_dice'])
            variations.append(scale)
        
        df = pd.DataFrame({
            'scale': variations,
            'mean_dice': mean_dices,
            'param_value': [base_params[param_idx] * s for s in variations]
        })
        sensitivity_results[param_name] = df
    
    return sensitivity_results

def plot_parameter_sensitivity(sensitivity_results, base_params, save_path=None):
    param_names = ['lambda', 'gamma', 'beta', 'sigmaFG', 'sigmaBG', 'alpha']
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle("Parameter Sensitivity Analysis", fontsize=16)
    axes = axes.flatten()
    
    for i, param_name in enumerate(param_names):
        ax = axes[i]
        df = sensitivity_results[param_name]
        
        ax.plot(df['param_value'], df['mean_dice'], 'o-', linewidth=2, markersize=8, color='blue')
        
        opt_val = base_params[i]
        opt_dice = df[df['param_value'] == opt_val]['mean_dice'].values
        if len(opt_dice) > 0:
            ax.axvline(x=opt_val, color='red', linestyle='--', label=f'Optimized ({opt_val:.3f})')
            ax.plot(opt_val, opt_dice[0], 'r*', markersize=12)
        
        ax.set_xlabel(f"{param_name} Value")
        ax.set_ylabel("Mean Dice Score")
        ax.set_title(f"Sensitivity to {param_name}")
        ax.grid(True, alpha=0.3)
        ax.legend()
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Sensitivity analysis plot saved to {save_path}")
    plt.show()
    plt.close()


def main():
    """Generate only experiments, tables and qualitative figures described in the paper."""
    if not os.path.isdir(DATASET_DIR):
        raise FileNotFoundError(f"Set DATASET_DIR to the LGG dataset directory: {DATASET_DIR}")
    all_pairs = collect_image_mask_pairs(DATASET_DIR)
    if not all_pairs:
        raise ValueError("No paired LGG images and masks found")
    train_pairs, val_pairs, train_patient_ids, val_patient_ids = split_by_patient(
        all_pairs, train_ratio=TRAIN_RATIO, seed=SEED)
    train_pos, _ = split_positive_negative_pairs(train_pairs)
    val_pos, val_neg = split_positive_negative_pairs(val_pairs)
    random.shuffle(train_pos)
    optimization_pairs = train_pos[:OPT_SUBSET_SIZE]
    if not optimization_pairs:
        raise ValueError("No tumor-positive training slices")

    bounds_low = [0.1, 0.1, 0.0001, 1., 1., 0.05]
    bounds_high = [20., 20., 0.05, 80., 80., 1.50]
    def fitness_func(theta):
        dices = []
        for image_path, mask_path in optimization_pairs:
            image, ground_truth = load_image_mask(image_path, mask_path, target_size=IMAGE_SIZE)
            prediction = graph_cut_segment(rgb_to_working_gray(image), theta)
            dices.append(dice_score(ground_truth, prediction))
        return 1.0 - float(np.mean(dices))

    opt = GSO(fitness_func=fitness_func, sol_count=SOL_COUNT, dimensions=6,
              iterations_count=ITERATIONS_COUNT, lower_bound=bounds_low,
              upper_bound=bounds_high, mutation_rate=MUTATION_RATE, verbose=True)
    params = np.asarray(opt.leader_solution, dtype=float)
    np.save(os.path.join(EXPERIMENT_DIR, "best_params.npy"), params)
    config = dict(seed=SEED, image_size=IMAGE_SIZE, train_ratio=TRAIN_RATIO,
                  optimization_subset_size=len(optimization_pairs),
                  train_patients=len(train_patient_ids), validation_patients=len(val_patient_ids),
                  population=SOL_COUNT, iterations=ITERATIONS_COUNT,
                  mutation_rate=MUTATION_RATE, bounds_lower=bounds_low, bounds_upper=bounds_high,
                  optimized_parameters=params.tolist())
    with open(os.path.join(EXPERIMENT_DIR, "config.json"), "w") as f:
        json.dump(config, f, indent=2)

    # Paper Tables 2, 3 and 4: LGG metrics, baseline comparison and ablation.
    positive_summary, positive_df = evaluate_pairs_detailed(val_pos, params, segment_fn=graph_cut_segment)
    baseline_summary, baseline_df = evaluate_pairs_detailed(val_pos, params, segment_fn=graph_cut_segment_baseline)
    ablation_summary, _ = run_ablation_study(val_pos, params)
    pd.DataFrame([positive_summary]).to_csv(os.path.join(EXPERIMENT_DIR, "table2_lgg_positive.csv"), index=False)
    pd.DataFrame([dict(method="Baseline", **baseline_summary),
                  dict(method="Final", **positive_summary)]).to_csv(
                      os.path.join(EXPERIMENT_DIR, "table3_baseline_vs_final.csv"), index=False)
    ablation_summary.to_csv(os.path.join(EXPERIMENT_DIR, "table4_ablation.csv"), index=False)

    # Paper: paired Wilcoxon signed-rank significance test.
    wilcoxon = wilcoxon_test_from_dfs(baseline_df, positive_df, metric="dice")
    with open(os.path.join(EXPERIMENT_DIR, "wilcoxon.json"), "w") as f:
        json.dump(wilcoxon, f, indent=2)

    # Paper Table 5: tumor-negative slice false positives.
    negative_summary = evaluate_negative_pairs(val_neg, params, segment_fn=graph_cut_segment)
    pd.DataFrame([negative_summary]).to_csv(os.path.join(EXPERIMENT_DIR, "table5_lgg_negative.csv"), index=False)

    # Paper Table 8: inference time per 128x128 image.
    runtime_base = benchmark_runtime(val_pos, params, graph_cut_segment_baseline, num_cases=20)
    runtime_final = benchmark_runtime(val_pos, params, graph_cut_segment, num_cases=20)
    pd.DataFrame([dict(method="Baseline", **runtime_base),
                  dict(method="Final", **runtime_final)]).to_csv(
                      os.path.join(EXPERIMENT_DIR, "table8_runtime.csv"), index=False)

    # Paper Figure 7, Figures 8-10, Figure 11.
    show_predictions(val_pos, params, n=5,
                     save_path=os.path.join(EXPERIMENT_DIR, "figure7_lgg_examples.png"),
                     segment_fn=graph_cut_segment)
    lookup = {im: mask for im, mask in val_pos}
    save_ranked_examples(positive_df, params,
                         save_dir=os.path.join(EXPERIMENT_DIR, "figures8_9_10_ranked"),
                         pairs_lookup=lookup, n_each=3, segment_fn=graph_cut_segment)
    save_false_positive_examples(val_neg, params,
                                 save_dir=os.path.join(EXPERIMENT_DIR, "figure11_false_positives"),
                                 n_examples=3, segment_fn=graph_cut_segment)

    # Paper Table 7 and Figure 6: 6-parameter sensitivity, using published scales.
    names = ['lambda', 'gamma', 'beta', 'sigmaFG', 'sigmaBG', 'alpha']
    sensitivity = run_parameter_sensitivity_analysis(val_pos, params.copy(), names, list(range(6)))
    pd.DataFrame({name: data['mean_dice'].values for name, data in sensitivity.items()},
                 index=next(iter(sensitivity.values()))['scale'].values).rename_axis('scale').to_csv(
                     os.path.join(EXPERIMENT_DIR, 'table7_parameter_sensitivity.csv'))
    plot_parameter_sensitivity(sensitivity, params,
                               save_path=os.path.join(EXPERIMENT_DIR, 'figure6_parameter_sensitivity.png'))

    # Paper Table 6: BraTS transfer with LGG-trained parameters, no tuning.
    if SECOND_DATASET_DIR and os.path.exists(SECOND_DATASET_DIR):
        brats_pos, brats_neg, brats_all, brats_pos_df = evaluate_second_dataset(SECOND_DATASET_DIR, params)
        if brats_pos is not None:
            pd.DataFrame([brats_pos]).to_csv(os.path.join(EXPERIMENT_DIR, 'table6_brats_transfer.csv'), index=False)
            # Figure 5: sample BraTS transfer predictions.
            processed = get_preprocessed_brats_path(SECOND_DATASET_DIR)
            if processed:
                pairs, _ = split_positive_negative_pairs(collect_image_mask_pairs(processed))
                create_brats_sample_predictions(pairs, params, n_samples=2,
                   save_path=os.path.join(EXPERIMENT_DIR, 'figure5_brats_examples.png'))
            # Section 5.6: optional re-optimization reported in the paper.
            if BRATS_OPTIMIZATION:
                brats_params = optimize_for_brats(params, SECOND_DATASET_DIR, EXPERIMENT_DIR)
                brats_opt_pos, brats_opt_neg, _ = evaluate_brats_full(
                    SECOND_DATASET_DIR, brats_params, EXPERIMENT_DIR)
                with open(os.path.join(EXPERIMENT_DIR, 'brats_reoptimization_comparison.json'), 'w') as f:
                    json.dump({'LGG_transfer': brats_pos, 'BraTS_reoptimized': brats_opt_pos}, f, indent=2)
    else:
        print('BraTS dataset not configured; skipping the paper BraTS experiments.')

    print('Paper experiment artifacts saved in:', EXPERIMENT_DIR)

if __name__ == "__main__":
    main()
